# MineAgent — Full Training Notebook
**RTX T4 (Kaggle Free)** | Llama 3.2:3b | QLoRA | Stage 1 + Stage 2

## Setup:
1. Add your `session_log.jsonl` as a Kaggle Input Dataset named `mineagent-session`
2. Add your HuggingFace token in Kaggle Secrets as `HF_TOKEN`
3. Enable GPU T4 x2 (or T4 single)
4. Run All Cells


In [1]:
import subprocess, sys

# Core training libs (Pinning trl<0.9.0 because newer versions break Unsloth's dynamic GKD compilation)
subprocess.run([sys.executable,'-m','pip','install','-q',
    'unsloth','trl<0.9.0','peft',
    'bitsandbytes','datasets>=2.14','accelerate',
    'huggingface_hub','mwclient','requests'], check=False)

print('Install done')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ── Cell 2: GPU + HF Login ────────────────────────────────────────
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,1),'GB')
print('BF16:', torch.cuda.is_bf16_supported())

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
login(HF_TOKEN)
print('HuggingFace: logged in')

GPU: Tesla T4
VRAM: 14.6 GB
BF16: True
HuggingFace: logged in


In [3]:
# ── Cell 3: Collect Training Data ───────────────────────────────
import json, time, random, requests, os
from pathlib import Path
from datasets import load_dataset

Path('/kaggle/working/data').mkdir(exist_ok=True)

SYS_KNOW = 'You are MineAgent, an expert Minecraft AI. Answer accurately about Minecraft.'
SYS_ACT  = 'You are MineAgent. Output ONLY valid JSON: {"action":"NAME","params":{},"reasoning":""}'

def qa_chat(q, a, sys=None):
    return {'messages':[{'role':'system','content':sys or SYS_KNOW},
                        {'role':'user','content':q},
                        {'role':'assistant','content':a}]}

def save(path, data):
    with open(path,'w') as f:
        [f.write(json.dumps(x)+'\n') for x in data]
    print(f'Saved {len(data)} -> {path}')

# ── Source 1: Minecraft Wiki ─────────────────────────────────────
def get_wiki(max_pages=400):
    API = 'https://minecraft.wiki/api.php'
    CATS = ['Blocks','Items','Crafting','Mobs','Biomes','Food',
            'Tools','Weapons','Armor','Mechanics']
    pairs, seen = [], set()
    for cat in CATS:
        if len(seen) >= max_pages: break
        try:
            r = requests.get(API, params={'action':'query','list':'categorymembers',
                'cmtitle':f'Category:{cat}','cmlimit':'100','format':'json'}, timeout=15)
            for p in r.json()['query']['categorymembers']:
                if p['title'] in seen or len(seen) >= max_pages: continue
                seen.add(p['title'])
                r2 = requests.get(API, params={'action':'query','titles':p['title'],
                    'prop':'extracts','explaintext':True,'format':'json'}, timeout=15)
                txt = list(r2.json()['query']['pages'].values())[0].get('extract','')[:1000]
                if len(txt) < 80: continue
                pairs.append(qa_chat(f"What is {p['title']} in Minecraft?", txt))
                pairs.append(qa_chat(f"How do you use {p['title']} in Minecraft?", txt[:500]))
                time.sleep(0.1)
        except Exception as e:
            pass
    print(f'Wiki: {len(pairs)} pairs')
    return pairs

# ── Source 2: Minecraft HF Datasets (Reliable replacements) ──────
def get_hf_qa(limit=15000):
    pairs = []
    try:
        # Use reliable public Minecraft Q&A dataset instead of broken MineDojo
        ds = load_dataset('naklecha/minecraft-question-answer-700k', split='train', streaming=True)
        for item in ds:
            if len(pairs) >= limit: break
            q = item.get('question','') or item.get('instruction','')
            a = item.get('answer','') or item.get('output','')
            if len(a) > 20:
                pairs.append(qa_chat(q, a[:500]))
        print(f'HF Q&A Dataset 1: {len(pairs)} pairs')
    except Exception as e:
        print(f'Dataset 1 error: {e}')
    
    try:
        ds2 = load_dataset('minhaozhang/minecraft-question-answer-630k', split='train', streaming=True)
        for item in ds2:
            if len(pairs) >= limit * 1.5: break
            q = item.get('question','') or item.get('instruction','')
            a = item.get('answer','') or item.get('output','')
            if len(a) > 20:
                pairs.append(qa_chat(q, a[:500]))
        print(f'HF Q&A Dataset 2: {len(pairs)} pairs')
    except Exception as e:
        print(f'Dataset 2 error: {e}')
        
    return pairs

# ── Source 3: Bot sessions (uploaded as Kaggle dataset input) ────
def get_sessions():
    path = '/kaggle/input/mineagent-session/session_log.jsonl'
    if not Path(path).exists():
        return []
    VALID = {'SEEK','MINE','MOVE','CRAFT','EAT','CHAT','FOLLOW','GOTO','STOP'}
    pairs = []
    for line in open(path):
        item = json.loads(line)
        if item.get('source') != 'llm': continue
        a = item.get('action',{})
        if a.get('action') not in VALID: continue
        q = (f"Goal:{item.get('goal','')} "
             f"HP:{item.get('player',{}).get('health',20)} "
             f"Inv:{list(item.get('inventory',{}).keys())[:5]} "
             f"Nearby:{[b['name'] for b in item.get('nearby_blocks',[])[:5]]} "
             f"Progress:{item.get('goal_progress','')}")
        out = json.dumps({'action':a['action'],'params':a.get('params',{}),'reasoning':a.get('reasoning','')})
        pairs.append(qa_chat(q, out, sys=SYS_ACT))
    print(f'Sessions: {len(pairs)} pairs')
    return pairs

# ── Build datasets ────────────────────────────────────────────────
stage1 = get_wiki(max_pages=400)
stage1 += get_hf_qa(limit=18000)

stage2 = get_sessions()

random.shuffle(stage1)
random.shuffle(stage2)

save('/kaggle/working/data/stage1.jsonl', stage1)
save('/kaggle/working/data/stage2.jsonl', stage2)
print(f'\nStage1: {len(stage1)} | Stage2: {len(stage2)}')

Wiki: 726 pairs


README.md:   0%|          | 0.00/943 [00:00<?, ?B/s]

HF Q&A Dataset 1: 18000 pairs


README.md:   0%|          | 0.00/697 [00:00<?, ?B/s]

HF Q&A Dataset 2: 27000 pairs
Saved 27726 -> /kaggle/working/data/stage1.jsonl
Saved 0 -> /kaggle/working/data/stage2.jsonl

Stage1: 27726 | Stage2: 0


In [ ]:
# ── Cell 4: Train Stage 1 (Knowledge) ────────────────────────────
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

MODEL_ID = 'unsloth/Llama-3.2-3B-Instruct'

# 1. Load Model
model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, 
    max_seq_length=1024,
    dtype=None, 
    load_in_4bit=True
)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    lora_dropout=0.0, bias='none')

tok.pad_token = tok.eos_token
tok.padding_side = 'right'

# 2. Load Dataset
ds1 = load_dataset('json', data_files={'train':'/kaggle/working/data/stage1.jsonl'}, split='train')
split1 = ds1.train_test_split(test_size=0.05, seed=42)
split1 = split1.map(lambda ex: {'text': tok.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)})

# 3. Train on Multiple GPUs
trainer1 = SFTTrainer(
    model=model, tokenizer=tok,
    train_dataset=split1['train'], eval_dataset=split1['test'],
    dataset_text_field='text', max_seq_length=512,
    args=TrainingArguments(
        output_dir='/kaggle/working/stage1',
        num_train_epochs=3,
        per_device_train_batch_size=32,   # up from 16 — fill the VRAM
        gradient_accumulation_steps=2,   # keep effective batch = 64
        gradient_checkpointing=True,      # trade compute for memory
        optim="adamw_8bit",              # 8-bit optimizer = less VRAM,
        learning_rate=3e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        fp16=True, bf16=False, 
        logging_steps=50,
        eval_strategy='steps', 
        eval_steps=100,
        save_strategy='no',
        report_to='none',
        ddp_find_unused_parameters=False, # Required for DDP
    ))

print('=== Stage 1 Training Start ===')
trainer1.train()
model.save_pretrained('/kaggle/working/stage1-lora')
tok.save_pretrained('/kaggle/working/stage1-lora')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-16 17:20:42.625414: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778952042.844321      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778952042.911597      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778952043.446871      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778952043.446907      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778952043.446910      57 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.8: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


=== Stage 1 Training Start ===


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26,339 | Num Epochs = 10 | Total steps = 2,060
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 2 x 1) = 128
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss


In [ ]:
# ── Cell 5: Train Stage 2 (Behavior Cloning) ─────────────────────
import gc
del trainer1; gc.collect(); torch.cuda.empty_cache()

model2, tok2 = FastLanguageModel.from_pretrained(
    model_name='/kaggle/working/stage1-lora',
    max_seq_length=1024, dtype=torch.float16, load_in_4bit=True)

model2 = FastLanguageModel.get_peft_model(
    model2, r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.0, bias='none')

tok2.pad_token = tok2.eos_token; tok2.padding_side = 'right'

# ── Load Stage 2 data ────────────────────────────────────────────
# Priority: 1) Bot sessions  2) Synthetic dataset (1500 unique examples)
stage2_path = '/kaggle/working/data/stage2.jsonl'
synth_path  = '/kaggle/input/mineagent-session/synthetic_stage2.jsonl'

stage2_size = Path(stage2_path).stat().st_size if Path(stage2_path).exists() else 0

if stage2_size < 500:  # no real session data
    if Path(synth_path).exists():
        import shutil
        shutil.copy(synth_path, stage2_path)
        lines = open(stage2_path).readlines()
        print(f'Loaded {len(lines)} synthetic examples (1500 unique, diverse scenarios)')
    else:
        print('ERROR: synthetic_stage2.jsonl not found!')
        print('Upload training/synthetic_stage2.jsonl to your Kaggle dataset named mineagent-session')
        raise FileNotFoundError('Missing synthetic_stage2.jsonl')
else:
    # Merge real sessions + synthetic for best results
    if Path(synth_path).exists():
        with open(stage2_path, 'a') as dst, open(synth_path) as src:
            dst.write(src.read())
        print('Merged real sessions + 1500 synthetic examples')
    lines = open(stage2_path).readlines()
    print(f'Total stage2 examples: {len(lines)}')

ds2 = load_dataset('json', data_files={'train': stage2_path}, split='train')
split2 = ds2.train_test_split(test_size=0.08, seed=42)
split2 = split2.map(lambda x: {'text': tok2.apply_chat_template(
    x['messages'], tokenize=False, add_generation_prompt=False)})

print(f'Train: {len(split2["train"])} | Val: {len(split2["test"])}')

trainer2 = SFTTrainer(
    model=model2, tokenizer=tok2,
    train_dataset=split2['train'], eval_dataset=split2['test'],
    dataset_text_field='text', max_seq_length=1024,
    args=TrainingArguments(
        output_dir='/kaggle/working/stage2',
        num_train_epochs=5,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        bf16=False, fp16=True, logging_steps=20,
        eval_strategy='steps', eval_steps=100,
        save_steps=200, save_total_limit=1,
        report_to='none',
    ))

print('=== Stage 2 Training Start ===')
trainer2.train()
model2.save_pretrained('/kaggle/working/mineagent-v1-lora')
tok2.save_pretrained('/kaggle/working/mineagent-v1-lora')
print('Stage 2 done. Loss:', trainer2.state.log_history[-1])

In [ ]:
# ── Cell 6: Export to GGUF ────────────────────────────────────────
import gc
del trainer2, model; gc.collect(); torch.cuda.empty_cache()

print('Converting to GGUF Q4_K_M...')
model2.save_pretrained_gguf(
    '/kaggle/working/mineagent-v1-gguf',
    tok2,
    quantization_method='q4_k_m'
)
import os
files = os.listdir('/kaggle/working/mineagent-v1-gguf')
print('GGUF files:', files)

In [ ]:
# ── Cell 7: Push to HuggingFace Hub ──────────────────────────────
# Model will be at: https://huggingface.co/YOUR_USERNAME/mineagent-v1
from huggingface_hub import HfApi
import subprocess

HF_USERNAME = 'YOUR_HF_USERNAME'  # ← CHANGE THIS
REPO_ID = f'{HF_USERNAME}/mineagent-v1'
api = HfApi()

# Create repo
api.create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True)

# Upload GGUF
api.upload_folder(
    folder_path='/kaggle/working/mineagent-v1-gguf',
    repo_id=REPO_ID,
    repo_type='model',
)

# Upload LoRA adapter too
api.upload_folder(
    folder_path='/kaggle/working/mineagent-v1-lora',
    repo_id=f'{HF_USERNAME}/mineagent-v1-lora',
    repo_type='model',
)

print(f'Model uploaded!')
print(f'GGUF: https://huggingface.co/{REPO_ID}')
print(f'LoRA: https://huggingface.co/{HF_USERNAME}/mineagent-v1-lora')

## ✅ Training Complete!
Your model is now on HuggingFace. Run `download_model.py` locally to pull it.